# 04 — Leakage-Aware Feature Engineering

Build calendar, operational, lag, rolling, momentum, quality, and seven-day
forecast-target fields. Historical rolling features are shifted by one day.

In [ ]:
from pathlib import Path
import sys
import numpy as np  # noqa: F401 -- shared setup; used by modeling notebooks
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "output"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

In [ ]:
from src.feature_engineering import (
    FeatureEngineeringConfig,
    build_feature_matrix,
    feature_manifest,
    split_features_and_target,
)

metrics = pd.read_csv(
    PROCESSED_DIR / "uac_capacity_metrics_daily.csv",
    parse_dates=["Date"],
).set_index("Date")
result = build_feature_matrix(
    metrics,
    FeatureEngineeringConfig(target_horizon_days=7),
)
manifest = feature_manifest(result)
print(
    {
        "rows": len(result.frame),
        "feature_columns": len(result.feature_columns),
        "target_columns": len(result.target_columns),
        "target_horizon_days": result.config.target_horizon_days,
    }
)
manifest.groupby(["Role", "Category"]).size().rename("columns")

In [ ]:
target_name = "target_total_load_t_plus_7d"
X, y = split_features_and_target(result, target_name, drop_missing=True)
print(
    {
        "complete_training_rows": len(X),
        "first_training_date": X.index.min().date().isoformat(),
        "last_training_date": X.index.max().date().isoformat(),
    }
)
manifest.sort_values(["Role", "Category", "Column"]).head(25)

In [ ]:
assert all(column not in X.columns for column in result.target_columns)
assert not np.isinf(X.select_dtypes(include=["number"]).to_numpy(dtype=float)).any()
print("Target leakage and non-finite-value checks passed.")

Do not select features using the test period. Fit imputers,
scalers, and models only on chronologically earlier training observations.